# 04 - Sensibilità all'orario del pasto e alla velocità di assorbimento (adult#001, 60 g)

**Obiettivo.** Come suggerito dalla co-relatrice, osservare come cambia la risposta dell'anello chiuso al variare
(a) dell'istante di assunzione del pasto e (b) della velocità di assorbimento dei carboidrati, a parità di quantità (60 g).

La velocità di assorbimento si modifica con `assorb_scale`, che scala insieme le costanti dello svuotamento gastrico
(`kmax`, `kmin`) e dell'assorbimento intestinale (`kabs`) del modello UVa/Padova: 0.5 è un pasto lento, 2 un pasto rapido.
Scalare solo `kabs` non basta: nel modello la dinamica del pasto è dominata dallo svuotamento gastrico.
La velocità di ingestione resta quella di simglucose (5 g/min).

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
import aid_sim as A, notebook_utils as U
pd.set_option("display.width", 220); pd.set_option("display.max_colwidth", 110)
SEED = 1                                  # seme del rumore del sensore (uguale per tutti i controllori)
COL = {"PID senza AW": "tab:gray", "PID": "tab:red", "PID cons.": "tab:orange",
       "PID+PLGS": "tab:green", "PID+IOB": "tab:purple", "PID+IOB+PLGS": "tab:blue"}
print("Versioni delle librerie usate:"); display(U.library_versions())

In [ ]:
def finestra(df):
    """Solo la finestra di analisi (08:00-24:00): l'assestamento 00:00-08:00 non entra né nei grafici né nelle metriche."""
    return df[df.t_h >= A.WARMUP_H]

def decora(ax, meal_h=A.MEAL_CLOCK_H, banda=True):
    """Linea del pasto e griglia; banda 70-180 mg/dL solo sui pannelli di glicemia."""
    if banda:
        ax.axhspan(70, 180, color="green", alpha=0.12)
    ax.axvline(meal_h, color="k", ls=":", lw=1.2); ax.grid(ls=":", alpha=0.5)

def tabella(risultati, chiavi):
    """risultati: dict {chiave: DataFrame}; chiavi: nomi delle colonne-indice per la tabella."""
    rows = []
    for k, df in risultati.items():
        k = k if isinstance(k, tuple) else (k,)
        rows.append({**dict(zip(chiavi, k)), **{m: round(float(v), 1) for m, v in A.metrics(df).items()}})
    return pd.DataFrame(rows)

In [ ]:
PAZIENTE = "adult#001"; CHO = 60.0
NOMI = ["PID", "PID+IOB+PLGS"]
display(U.scenario_table(PAZIENTE, CHO, SEED))

### Unità di misura delle variabili salvate
| Variabile | Unità | Significato |
|---|---|---|
| `t_h` | h | ora del giorno (la simulazione parte alle 00:00) |
| `CGM` | mg/dL | misura del sensore, con rumore: è l'unica grandezza vista dal controllore |
| `BG` | mg/dL | glicemia plasmatica del modello UVa/Padova (Gp/Vg), usata per le metriche |
| `Gsub` | mg/dL | glicemia sottocutanea senza rumore (`info['bg']` di simglucose) |
| `Insulin_Uh` | U/h | velocità di infusione comandata |
| `IOB` | U | insulina attiva netta stimata dal controllore (eccesso rispetto al basale) |
| `IOB_vero` | U | insulina sottocutanea netta del modello (Isc1 + Isc2 meno l'equilibrio basale) |
| `Vincolo` | - | vincolo attivo nel passo: `iob`, `plgs`, `sat_alta`, `sat_bassa` o vuoto |

Metriche: TIR, TAR, TBR in % del tempo nella finestra 08:00-24:00; `TIR_post6h` nelle 6 h dopo il pasto;
`TBR_tardivo` dalle 3 h dopo il pasto alle 24:00; picco e nadir in mg/dL; `insulina_tot` in U nella finestra.

## 1. Orario del pasto

In [ ]:
ORARI = [10.0, 12.0, 14.0, 16.0]
Ro = {(f"{int(h):02d}:00", n): A.simulate(PAZIENTE, CHO, A.CFG[n], seed=SEED, meal_clock_h=h)[0] for h in ORARI for n in NOMI}
fig, ax = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for j, n in enumerate(NOMI):
    for h in ORARI:
        d = finestra(Ro[(f"{int(h):02d}:00", n)]); ax[j].plot(d.t_h, d.BG, label=f"pasto alle {int(h)}:00")
    ax[j].axhspan(70, 180, color="green", alpha=0.12); ax[j].grid(ls=":", alpha=0.5); ax[j].set_title(n); ax[j].legend(fontsize=8)
    ax[j].set_xlabel("Ora del giorno [h]")
ax[0].set_ylabel("BG plasmatica [mg/dL]"); plt.tight_layout(); plt.show()
rows = []
for (o, n), df in Ro.items():
    m = A.metrics(df, meal_h=float(o[:2]))
    rows.append({"Orario": o, "Controllore": n, **{k: round(float(v), 1) for k, v in m.items()}})
display(pd.DataFrame(rows))

**Attenzione.** Con un pasto più tardo resta meno tempo prima delle 24:00: il `TBR_tardivo` di un pasto alle 16:00 copre
solo 5 ore. Per confrontare gli orari conviene guardare soprattutto `TIR_post6h`, picco e nadir.

## 2. Velocità di assorbimento

In [ ]:
SCALE = [0.5, 1.0, 2.0]
Ra = {(s, n): A.simulate(PAZIENTE, CHO, A.CFG[n], seed=SEED, assorb_scale=s)[0] for s in SCALE for n in NOMI}
fig, ax = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for j, n in enumerate(NOMI):
    for s in SCALE:
        d = finestra(Ra[(s, n)]); ax[j].plot(d.t_h, d.BG, label=f"assorbimento x {s:g}")
    decora(ax[j]); ax[j].set_title(n); ax[j].legend(fontsize=8); ax[j].set_xlabel("Ora del giorno [h]")
ax[0].set_ylabel("BG plasmatica [mg/dL]"); plt.tight_layout(); plt.show()
display(tabella(Ra, ["Assorbimento x", "Controllore"]))